# 11 — Evaluation

Compare the system's dot assignments against the manually labeled ground truth
using B-cubed Precision, Recall and F1.

- **Precision** — for each article, what fraction of its system-dot neighbours share the same GT label? Measures dot purity. Low when system under-splits.
- **Recall** — for each article, what fraction of its GT-cluster neighbours ended up in the same system dot? Measures grouping completeness. Low when system over-splits.
- **F1** — harmonic mean of precision and recall.

**Data:**
- Ground truth: `data/evaluation/ground_truth_*.json` — 24 manually labeled stories across 5 buckets
- System output: `data/processed/dots_louvain_tuned.pkl`

In [22]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("../").resolve()
EVAL_DIR = REPO_ROOT / "data" / "evaluation"

## Load Data

In [23]:
df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

with open(REPO_ROOT / "data/processed/stories.pkl", "rb") as f:
    stories = pickle.load(f)

with open(REPO_ROOT / "data/processed/dots_louvain_tuned.pkl", "rb") as f:
    all_dots = pickle.load(f)

print(f"Articles : {len(df):,}")
print(f"Stories  : {len(stories):,}")

Articles : 30,462
Stories  : 18,473


In [24]:
# load all ground truth buckets
BUCKETS = ["2_5", "5_10", "10_20", "20_50", "50+"]
BUCKET_LABELS = {"2_5": "2-5", "5_10": "5-10", "10_20": "10-20", "20_50": "20-50", "50+": "50+"}

ground_truth = {}
for bucket in BUCKETS:
    with open(EVAL_DIR / f"ground_truth_{bucket}.json") as f:
        data = json.load(f)
    for sid_str, v in data.items():
        ground_truth[int(sid_str)] = {**v, "bucket": BUCKET_LABELS[bucket]}

print(f"Ground truth stories: {len(ground_truth)}")

Ground truth stories: 24


## Helper Functions

In [25]:
url_to_idx = {row["url"]: i for i, row in df.iterrows()}


def system_labels_for_story(sid: int, gt_urls: list) -> list:
    """Convert system dots → per-article labels, matched by URL to ground truth order."""
    idx_to_dot = {}
    for dot_id, dot in enumerate(all_dots[sid]):
        for art_idx in dot["indices"]:
            idx_to_dot[art_idx] = dot_id
    return [idx_to_dot.get(url_to_idx.get(url), -1) for url in gt_urls]


def bcubed(gt_labels: list, sys_labels: list) -> tuple:
    """B-cubed precision, recall, F1."""
    gt = np.array(gt_labels)
    sys = np.array(sys_labels)
    n = len(gt)

    precisions, recalls = [], []
    for i in range(n):
        same_sys = sys == sys[i]
        same_gt = gt == gt[i]

        p_i = np.sum(same_sys & same_gt) / np.sum(same_sys)
        r_i = np.sum(same_sys & same_gt) / np.sum(same_gt)

        precisions.append(p_i)
        recalls.append(r_i)

    p = float(np.mean(precisions))
    r = float(np.mean(recalls))
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return round(p, 3), round(r, 3), round(f1, 3)

## Per-Story Evaluation

In [26]:
rows = []
for sid, gt in ground_truth.items():
    gt_labels = gt["labels"]
    sys_labels = system_labels_for_story(sid, gt["articles"])
    p, r, f1 = bcubed(gt_labels, sys_labels)

    rows.append({
        "story_id": sid,
        "bucket": gt["bucket"],
        "n_articles": len(gt_labels),
        "gt_dots": len(set(gt_labels)),
        "sys_dots": len(set(sys_labels)),
        "Precision": p,
        "Recall": r,
        "F1": f1,
    })

results = pd.DataFrame(rows).sort_values(["bucket", "story_id"])
results

,story_id,bucket,n_articles,gt_dots,sys_dots,Precision,Recall,F1
13,4738,10-20,18,4,4,1.000,1.000,1.000
11,6585,10-20,12,3,4,1.000,0.750,0.857
10,7809,10-20,11,5,6,0.818,0.758,0.787
14,8716,10-20,10,1,2,1.000,0.580,0.734
12,11154,10-20,14,3,4,1.000,0.732,0.845
3,265,2-5,2,1,1,1.000,1.000,1.000
2,2810,2-5,3,3,3,1.000,1.000,1.000
1,9785,2-5,5,2,2,1.000,1.000,1.000
4,15864,2-5,2,1,1,1.000,1.000,1.000
0,17716,2-5,2,1,1,1.000,1.000,1.000


## Summary Per Bucket

In [27]:
summary = (
    results.groupby("bucket")[["Precision", "Recall", "F1"]]
    .mean()
    .round(3)
    .reindex(["2-5", "5-10", "10-20", "20-50", "50+"])
)
summary.loc["Overall"] = results[["Precision", "Recall", "F1"]].mean().round(3)
summary

,Precision,Recall,F1
bucket,,,
2-5,1.000,1.000,1.000
5-10,0.962,0.695,0.783
10-20,0.964,0.764,0.845
20-50,0.788,0.880,0.826
50+,0.864,0.810,0.834
Overall,0.918,0.831,0.859


In [18]:
summary = (
    results.groupby("bucket")[["Precision", "Recall", "F1"]]
    .mean()
    .round(3)
    .reindex(["2-5", "5-10", "10-20", "20-50", "50+"])
)
summary.loc["Overall"] = results[["Precision", "Recall", "F1"]].mean().round(3)
summary

,Precision,Recall,F1
bucket,,,
2-5,1.000,1.000,1.000
5-10,1.000,0.695,0.794
10-20,0.976,0.714,0.820
20-50,0.852,0.866,0.855
50+,0.808,0.763,0.782
Overall,0.932,0.810,0.853


## Per-Story Detail

For each story: show ground truth labels vs system labels side by side.

In [20]:
def show_comparison(sid: int):
    gt = ground_truth[sid]
    gt_labels = gt["labels"]
    sys_labels = system_labels_for_story(sid, gt["articles"])
    p, r, f1 = bcubed(gt_labels, sys_labels)

    print(f"Story {sid}  [{gt['bucket']}]  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  (gt_dots={len(set(gt_labels))}  sys_dots={len(set(sys_labels))})")
    print(f"{'[i]':>4}  {'GT':>3}  {'SYS':>4}  title")
    print("-" * 80)
    for i, (url, gt_l, sys_l) in enumerate(zip(gt["articles"], gt_labels, sys_labels)):
        art_idx = url_to_idx.get(url)
        row = df.iloc[art_idx]
        pub = row["published_at_dt"].strftime("%d %b %H:%M")
        match = "✓" if gt_l == sys_l else "✗"
        print(f"  [{i:2}]  {gt_l:>3}  {sys_l:>4}    {match}   {pub} [{row['source']}] {row['title'][:50]}")

In [21]:
for sid in ground_truth:
    show_comparison(sid)
    print()

Story 17716  [2-5]  P=1.000  R=1.000  F1=1.000  (gt_dots=1  sys_dots=1)
 [i]   GT   SYS  title
--------------------------------------------------------------------------------
  [ 0]    0     0    ✓   20 May 14:09 [24chasa] Кремъл: Политиците от балтийските държави поддържа
  [ 1]    0     0    ✓   20 May 18:09 [fakti] Говори Москва: Политиците от балтийските държави п

Story 9785  [2-5]  P=1.000  R=1.000  F1=1.000  (gt_dots=2  sys_dots=2)
 [i]   GT   SYS  title
--------------------------------------------------------------------------------
  [ 0]    0     0    ✓   06 May 09:50 [bta] Джорджа Мелони ще се срещне в Рим с новоизбрания у
  [ 1]    0     0    ✓   06 May 09:59 [fakti] Мелони приема новия унгарски премиер Петер Мадяр в
  [ 2]    1     1    ✓   07 May 15:22 [nova] Петер Мадяр се срещна с Джорджа Мелони
  [ 3]    1     1    ✓   07 May 17:00 [vesti] Нова ера в Будапеща: Мадяр начерта бъдещето на Унг
  [ 4]    1     1    ✓   07 May 17:10 [standartnews] Нова ера в Унгария. Мадяр 